# Kaggle Predict F1 Pit Stops: Challenger Models and Feature Importance

A focused challenger notebook for CatBoost comparison, prediction diversity checks, and tree-model feature importance. CNN and RealMLP are intentionally deprioritized based on prior results and runtime friction.


## 1. Setup


In [ ]:
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "axes.labelsize": 10, "legend.frameon": False})

RANDOM_STATE = 42
TARGET = "PitNextLap"
ID_COL = "id"

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler


def evaluate_predictions(y_true, y_pred) -> dict:
    y_pred = np.clip(y_pred, 1e-6, 1 - 1e-6)
    return {
        "roc_auc": roc_auc_score(y_true, y_pred),
        "average_precision": average_precision_score(y_true, y_pred),
        "log_loss": log_loss(y_true, y_pred),
    }


def get_feature_columns(df: pd.DataFrame, target: str = TARGET, id_col: str = ID_COL):
    features = df.drop(columns=[c for c in [target, id_col] if c in df.columns])
    cat_cols = features.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in features.columns if c not in cat_cols]
    return num_cols, cat_cols


def make_tree_preprocessor(df: pd.DataFrame) -> ColumnTransformer:
    num_cols, cat_cols = get_feature_columns(df)
    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ], remainder="drop")
from lightgbm import LGBMClassifier

try:
    from catboost import CatBoostClassifier, Pool
    CATBOOST_AVAILABLE = True
except Exception as exc:
    CATBOOST_AVAILABLE = False
    print("CatBoost unavailable:", exc)

RUN_FAST = True
FAST_SAMPLE_SIZE = 120_000
N_SPLITS = 3 if RUN_FAST else 5
SELECTED_FEATURE_SET = "safe_plus_ratios_no_driver"


## 2. Load Data


In [ ]:
def find_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/playground-series-s6e5"),
        Path("/kaggle/input/playground-series-s6e5"),
        Path("../input/competitions/playground-series-s6e5"),
        Path("../input/playground-series-s6e5"),
        Path("data"),
        Path("../data"),
        Path("."),
    ]
    for path in candidates:
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            return path
    raise FileNotFoundError("Could not find train.csv and test.csv. Update DATA_DIR manually.")


def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        dtype = out[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="integer")
        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")
        elif pd.api.types.is_object_dtype(dtype):
            nunique = out[col].nunique(dropna=False)
            if nunique / max(len(out), 1) < 0.5:
                out[col] = out[col].astype("category")
    return out


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = reduce_memory_usage(pd.read_csv(DATA_DIR / "train.csv"))
test = reduce_memory_usage(pd.read_csv(DATA_DIR / "test.csv"))
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"DATA_DIR: {DATA_DIR}")
print(f"train: {train.shape}")
print(f"test: {test.shape}")
print(f"target positive rate: {train[TARGET].mean():.5f}")
train.head()

if RUN_FAST and len(train) > FAST_SAMPLE_SIZE:
    train_eval = (train.groupby(TARGET, group_keys=False)
        .apply(lambda x: x.sample(frac=min(1.0, FAST_SAMPLE_SIZE / len(train)), random_state=RANDOM_STATE))
        .sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True))
else:
    train_eval = train.copy()
print("train_eval:", train_eval.shape)


## 3. Shared Feature Set


In [ ]:
def add_features(df: pd.DataFrame, include_safe: bool = True, include_ratios: bool = True) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6

    if include_safe and {"LapNumber", "RaceProgress"}.issubset(out.columns):
        out["EstimatedRaceLaps"] = out["LapNumber"] / out["RaceProgress"].clip(lower=eps)
        out["EstimatedLapsRemaining"] = out["EstimatedRaceLaps"] - out["LapNumber"]
        out["LapNumber_x_RaceProgress"] = out["LapNumber"] * out["RaceProgress"]

    if include_ratios and {"TyreLife", "LapNumber"}.issubset(out.columns):
        out["TyreLife_to_LapNumber"] = out["TyreLife"] / out["LapNumber"].clip(lower=eps)

    if include_ratios and {"TyreLife", "EstimatedRaceLaps"}.issubset(out.columns):
        out["TyreLife_to_EstimatedRaceLaps"] = out["TyreLife"] / out["EstimatedRaceLaps"].clip(lower=eps)

    if include_safe and {"LapTime (s)", "LapTime_Delta"}.issubset(out.columns):
        out["LapTime_plus_Delta"] = out["LapTime (s)"] + out["LapTime_Delta"]
        out["AbsLapTime_Delta"] = out["LapTime_Delta"].abs()

    if include_safe and {"Position", "Position_Change"}.issubset(out.columns):
        out["PreviousPositionApprox"] = out["Position"] - out["Position_Change"]
        out["AbsPosition_Change"] = out["Position_Change"].abs()

    if include_safe and "Compound" in out.columns:
        compound = out["Compound"].astype(str)
        out["IsSoft"] = (compound == "SOFT").astype("int8")
        out["IsMedium"] = (compound == "MEDIUM").astype("int8")
        out["IsHard"] = (compound == "HARD").astype("int8")
        out["IsWetOrIntermediate"] = compound.isin(["WET", "INTERMEDIATE"]).astype("int8")

    return reduce_memory_usage(out)

train_model = add_features(train_eval)
test_model = add_features(test)
if SELECTED_FEATURE_SET.endswith("no_driver"):
    train_model = train_model.drop(columns=[c for c in ["Driver"] if c in train_model.columns])
    test_model = test_model.drop(columns=[c for c in ["Driver"] if c in test_model.columns])

X = train_model.drop(columns=[TARGET, ID_COL])
y = train_model[TARGET].astype("int8")
X_test = test_model.drop(columns=[ID_COL])
num_cols, cat_cols = get_feature_columns(train_model)
print("Features:", X.shape[1], "Categorical:", cat_cols)


## 4. CatBoost and LightGBM Comparison


In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

lgbm_params = {
    "objective": "binary", "n_estimators": 900 if RUN_FAST else 1800, "learning_rate": 0.02,
    "num_leaves": 127, "min_child_samples": 150, "subsample": 0.95, "colsample_bytree": 0.9,
    "reg_alpha": 1.0, "reg_lambda": 10.0, "max_bin": 255, "random_state": RANDOM_STATE,
    "n_jobs": -1, "verbose": -1,
}

catboost_params = {
    "loss_function": "Logloss", "eval_metric": "AUC", "iterations": 1200 if RUN_FAST else 2500,
    "learning_rate": 0.035, "depth": 7, "l2_leaf_reg": 6.0, "random_seed": RANDOM_STATE,
    "allow_writing_files": False, "verbose": False, "thread_count": -1,
}

def cv_lgbm_with_importance():
    oof = np.zeros(len(X), dtype=np.float32)
    rows, imp_rows = [], []
    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        X_train_raw, X_valid_raw = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        preprocessor = make_tree_preprocessor(X_train_raw)
        X_train = preprocessor.fit_transform(X_train_raw)
        X_valid = preprocessor.transform(X_valid_raw)
        feature_names = [name.split("__", 1)[-1] for name in preprocessor.get_feature_names_out()]
        model = LGBMClassifier(**lgbm_params)
        model.fit(X_train, y_train)
        pred = model.predict_proba(X_valid)[:, 1]
        oof[valid_idx] = pred
        rows.append({"model": "lightgbm", "fold": fold, **evaluate_predictions(y_valid, pred)})
        for feature, gain in zip(feature_names, model.booster_.feature_importance("gain")):
            imp_rows.append({"model": "lightgbm", "fold": fold, "feature": feature, "importance": gain})
    rows.append({"model": "lightgbm", "fold": "oof", **evaluate_predictions(y, oof)})
    return oof, rows, pd.DataFrame(imp_rows)

lgbm_oof, lgbm_rows, lgbm_importance = cv_lgbm_with_importance()
all_rows = lgbm_rows.copy()
catboost_importance = pd.DataFrame()
catboost_oof = None

if CATBOOST_AVAILABLE:
    X_cb = X.copy()
    for col in cat_cols:
        X_cb[col] = X_cb[col].astype(str).fillna("missing")
    cat_features = [X_cb.columns.get_loc(col) for col in cat_cols]
    catboost_oof = np.zeros(len(X_cb), dtype=np.float32)
    imp_rows = []
    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_cb, y), start=1):
        train_pool = Pool(X_cb.iloc[train_idx], y.iloc[train_idx], cat_features=cat_features)
        valid_pool = Pool(X_cb.iloc[valid_idx], y.iloc[valid_idx], cat_features=cat_features)
        model = CatBoostClassifier(**catboost_params)
        model.fit(train_pool, eval_set=valid_pool, use_best_model=True, early_stopping_rounds=120)
        pred = model.predict_proba(valid_pool)[:, 1]
        catboost_oof[valid_idx] = pred
        all_rows.append({"model": "catboost", "fold": fold, **evaluate_predictions(y.iloc[valid_idx], pred)})
        for feature, importance in zip(X_cb.columns, model.get_feature_importance(train_pool, type="PredictionValuesChange")):
            imp_rows.append({"model": "catboost", "fold": fold, "feature": feature, "importance": importance})
    all_rows.append({"model": "catboost", "fold": "oof", **evaluate_predictions(y, catboost_oof)})
    catboost_importance = pd.DataFrame(imp_rows)

results = pd.DataFrame(all_rows)
results[results["fold"].eq("oof")].sort_values("roc_auc", ascending=False)


## 5. Prediction Diversity Check


In [ ]:
if catboost_oof is not None:
    diversity = pd.DataFrame({
        "metric": ["prediction_correlation", "mean_abs_prediction_delta"],
        "value": [
            np.corrcoef(lgbm_oof, catboost_oof)[0, 1],
            np.mean(np.abs(lgbm_oof - catboost_oof)),
        ],
    })
    display(diversity)

    blend_rows = []
    for w in np.linspace(0, 1, 11):
        pred = w * lgbm_oof + (1 - w) * catboost_oof
        blend_rows.append({"lgbm_weight": w, "catboost_weight": 1 - w, **evaluate_predictions(y, pred)})
    catboost_blend_results = pd.DataFrame(blend_rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
    display(catboost_blend_results.head(10))
else:
    print("CatBoost OOF predictions unavailable; diversity check skipped.")


## 6. Non-Tree Model Decision


In [ ]:
non_tree_decision = pd.DataFrame([
    {
        "candidate": "CNN",
        "decision": "deprioritized",
        "reason": "Earlier baseline was materially behind boosted trees and adds preprocessing/runtime complexity.",
    },
    {
        "candidate": "RealMLP",
        "decision": "deprioritized",
        "reason": "Requires pytabkit in the runtime and has no evidence yet of beating tuned LightGBM.",
    },
])
non_tree_decision


## 7. Feature Importance


In [ ]:
lgbm_imp = lgbm_importance.groupby("feature", as_index=False).agg(importance=("importance", "mean")).sort_values("importance", ascending=False)
display(lgbm_imp.head(25))
fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(data=lgbm_imp.head(25), y="feature", x="importance", color=sns.color_palette("viridis", 8)[4], ax=ax)
ax.set_title("LightGBM feature importance, mean gain across folds")
ax.set_ylabel("")
plt.show()

if not catboost_importance.empty:
    cat_imp = catboost_importance.groupby("feature", as_index=False).agg(importance=("importance", "mean")).sort_values("importance", ascending=False)
    display(cat_imp.head(25))
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.barplot(data=cat_imp.head(25), y="feature", x="importance", color=sns.color_palette("viridis", 8)[5], ax=ax)
    ax.set_title("CatBoost feature importance, mean across folds")
    ax.set_ylabel("")
    plt.show()


## 8. Challenger Summary

CatBoost is useful as a sanity-check challenger, but the current evidence still favors tuned LightGBM. Continue with CatBoost only if future OOF prediction-diversity or leaderboard tests justify the added complexity.
